In [4]:
import pandas as pd

files = [
    ("medical_datasets/whole_multicare_dataset/data_dictionary.csv", "csv"),
    ("medical_datasets/whole_multicare_dataset/metadata.parquet", "parquet"),
    ("medical_datasets/whole_multicare_dataset/abstracts.parquet", "parquet"),
    ("medical_datasets/whole_multicare_dataset/cases.parquet", "parquet"),
    ("medical_datasets/whole_multicare_dataset/case_images.parquet", "parquet"),
    ("medical_datasets/whole_multicare_dataset/captions_and_labels.csv", "csv"),
]

for filename, ftype in files:
    try:
        if ftype == "parquet":
            df = pd.read_parquet(filename)
        else:
            df = pd.read_csv(filename, low_memory=False)

        print("=" * 60)
        print(f"File: {filename}")
        print(f"Total Rows: {len(df):,}")
        print(f"Total Columns: {len(df.columns)}")
        print("Columns:")
        for col in df.columns:
            print(f"  - {col}")
    except Exception as e:
        print(f"Error reading {filename}: {e}")

File: medical_datasets/whole_multicare_dataset/data_dictionary.csv
Total Rows: 45
Total Columns: 3
Columns:
  - file
  - field
  - explanation
File: medical_datasets/whole_multicare_dataset/metadata.parquet
Total Rows: 76,137
Total Columns: 2
Columns:
  - article_id
  - article_metadata
File: medical_datasets/whole_multicare_dataset/abstracts.parquet
Total Rows: 76,137
Total Columns: 2
Columns:
  - abstract
  - article_id
File: medical_datasets/whole_multicare_dataset/cases.parquet
Total Rows: 76,137
Total Columns: 2
Columns:
  - cases
  - article_id
File: medical_datasets/whole_multicare_dataset/case_images.parquet
Total Rows: 76,137
Total Columns: 2
Columns:
  - case_images
  - article_id
File: medical_datasets/whole_multicare_dataset/captions_and_labels.csv
Total Rows: 139,254
Total Columns: 17
Columns:
  - file_id
  - file
  - main_image
  - image_component
  - patient_id
  - license
  - file_size
  - caption
  - case_substring
  - image_type
  - image_subtype
  - radiology_region


# A comprehensive guide of the Filter

In [ ]:
comprehensive_filter_list = [
    # --------------------------------------------------------------------------
    # 1. ARTICLE-LEVEL FILTERS
    # --------------------------------------------------------------------------
    {
        # Minimum publication year (dataset covers 1990 to 2024)
        'field': 'min_year',
        'string_list': ['2015']  # Example: only articles published from 2015 onward[cite: 1, 2]
    },
    {
        # Maximum publication year
        'field': 'max_year',
        'string_list': ['2023']  # Example: articles published up to 2023
    },
    {
        # Article license filtering
        # Commercial options: 'CC0', 'CC BY', 'CC BY-SA', 'CC BY-ND'[cite: 2]
        # Non-commercial options: 'CC BY-NC', 'CC BY-NC-SA', 'CC BY-NC-ND'[cite: 2]
        # Other: 'author_manuscript', 'NO-CC CODE'[cite: 2]
        'field': 'license',
        'string_list': ['CC BY', 'CC BY-NC', 'CC BY-NC-SA']  # Include only these specific licenses[cite: 1, 2]
    },
    {
        # Keywords from article metadata (~87k unique terms in mdc.keyword_list)[cite: 2]
        # 'operator':
        #   - 'all' (default): article must have ALL keywords in string_list[cite: 2]
        #   - 'any': article metadata has at least ONE of the keywords[cite: 2]
        #   - 'none': excludes articles containing ANY of these keywords[cite: 2]
        # 'match_type':
        #   - 'full_match' (default): exact string equality[cite: 2]
        #   - 'partial_match': substring match (e.g., 'cardio' matches 'cardiovascular')[cite: 2]
        'field': 'keywords',
        'string_list': ['oncology', 'metastasis', 'glioblastoma'],
        'operator': 'any',
        'match_type': 'partial_match'
    },
    {
        # Medical Subject Headings (~38k terms in mdc.mesh_term_list)[cite: 2]
        # Supports the same 'operator' ('all', 'any', 'none') and 'match_type' ('full_match', 'partial_match')[cite: 2]
        'field': 'mesh_terms',
        'string_list': ['Magnetic Resonance Imaging', 'Neoplasms'],
        'operator': 'all',
        'match_type': 'full_match'
    },

    # --------------------------------------------------------------------------
    # 2. PATIENT-LEVEL (DEMOGRAPHIC) FILTERS
    # --------------------------------------------------------------------------
    {
        # Minimum patient age in years (patients < 1 y.o. are coded as 0)[cite: 1, 2]
        'field': 'min_age',
        'string_list': ['18']  # Filter for adult cohort[cite: 1, 2]
    },
    {
        # Maximum patient age in years
        'field': 'max_age',
        'string_list': ['75']  # Exclude geriatric patients above 75
    },
    {
        # Patient gender: 'Female', 'Male', 'Transgender', 'Unknown'[cite: 1, 2]
        # Pass multiple to include more than one demographic class
        'field': 'gender',
        'string_list': ['Male', 'Female']  # Exclude 'Unknown' and 'Transgender'[cite: 2]
    },

    # --------------------------------------------------------------------------
    # 3. CLINICAL CASE TEXT FILTERS
    # --------------------------------------------------------------------------
    {
        # Substring search across full clinical case presentation narratives[cite: 1, 2]
        # Always case-insensitive[cite: 2]
        # 'operator':
        #   - 'any': narrative mentions at least one term[cite: 2]
        #   - 'all': narrative must contain all listed terms[cite: 2]
        #   - 'none': negative assertion; exclude cases mentioning these terms[cite: 2]
        'field': 'case_strings',
        'string_list': ['headache', 'seizure', 'neurological deficit'],
        'operator': 'any'
    },

    # --------------------------------------------------------------------------
    # 4. IMAGE-LEVEL FILTERS (CAPTIONS & LABELS)
    # --------------------------------------------------------------------------
    {
        # Substring search across individual sub-image captions[cite: 1, 2]
        # 'operator': 'any', 'all', or 'none'[cite: 2]
        # 'matching_case':
        #   - False (default): case-insensitive search[cite: 2]
        #   - True: strict case-sensitive matching (useful for acronyms like 'AP', 'PA', 'T1', 'T2')[cite: 2]
        'field': 'caption',
        'string_list': ['axial', 'contrast-enhanced', 'edema'],
        'operator': 'all',
        'matching_case': False
    },
    {
        # Standardized taxonomy labels from mdc.image_label_list (141+ categories)[cite: 1, 2]
        # Covers:
        #   - Modalities / Subtypes: 'ct', 'mri', 'x_ray', 'ultrasound', 'h&e', 'ihc', etc.[cite: 1, 2]
        #   - Anatomical Regions: 'head', 'thorax', 'abdomen', 'pelvis', 'spine', etc.[cite: 1, 2]
        #   - Views / Planes: 'axial', 'sagittal', 'frontal', 'coronal', 'oblique', etc.[cite: 1, 2]
        #   - Specific sequences/stains: 't1', 't2', 'flair', 'dwi', 'masson_trichrome', etc.[cite: 1, 2]
        # 'operator':
        #   - 'all': image must have all listed labels (e.g., must be both 'mri' AND 'head')[cite: 2]
        #   - 'any': image matches if it has at least one of these labels[cite: 2]
        #   - 'none': explicitly exclude images tagged with these labels (e.g., negative filter for artifacts/charts)[cite: 2]
        'field': 'label',
        'string_list': ['mri', 'head', 'axial', 't1'],
        'operator': 'all'
    }
]

In [ ]:
# 1. 'multimodal' (Default): Extracts paired text (cases.csv) and sub-images (images/ folder)[cite: 2]
mdc.create_dataset( # type: ignore
    dataset_name='demo_multimodal_subset',
    filter_list=comprehensive_filter_list,
    dataset_type='multimodal'
)

# 2. 'text': Extracts only clinical case texts and metadata (no images copied)[cite: 1, 2]
# dataset_type='text'

# 3. 'image': Extracts image folders categorized by metadata (image_metadata.json)[cite: 1, 2]
# dataset_type='image'

# 4. 'case_series': Groups images into dedicated subfolders per patient ID[cite: 1, 2]
# dataset_type='case_series'

In [1]:
from multiversity.multicare_dataset import MedicalDatasetCreator
import os
import pandas as pd
mdc = MedicalDatasetCreator(directory = 'medical_datasets')

The MultiCaRe Dataset is already downloaded.
Importing and pre-processing the main files.
Done!


In [2]:
mdc.mesh_term_list

['Cardiovascular Agents / economics',
 'Vitreoretinopathy, Proliferative / diagnosis',
 'Cryoglobulinemia / drug therapy',
 'Diverticulum, Esophageal / surgery',
 'Fatty Acids, Omega-3 / pharmacology',
 'Irinotecan',
 'Skull Neoplasms / genetics',
 'Cloning, Molecular',
 'Robotics / methods',
 'Trichosporonosis / drug therapy',
 'Physicians / standards',
 'Monocytes / immunology',
 'Uterus / microbiology',
 'Skin Neoplasms / pathology',
 'Linoleic Acid / analysis',
 'Community-Institutional Relations / standards',
 'Ear Canal / drug effects',
 'Masseter Muscle / abnormalities',
 'Molecular Biology',
 'Hypopituitarism / diagnostic imaging',
 'Subacute Combined Degeneration / pathology',
 'Anthelmintics / administration & dosage',
 'Leprosy, Multibacillary',
 'Asthma / immunology',
 'Esophagus / injuries',
 'Proline / analogs & derivatives',
 'Spike Glycoprotein, Coronavirus',
 'Tracheostomy / instrumentation',
 'Comamonas',
 'Angiolymphoid Hyperplasia with Eosinophilia / etiology',
 'Mu

In [3]:
mdc.keyword_list

['',
 'abdominal incisional hernia',
 'scrotal hematoma',
 'guillain–barre syndrome',
 'parent artery occlusion',
 'type iii hyperlipoproteinemia',
 'protein hydrolysates',
 'sural nerve graft',
 'milium',
 'work identity',
 'deep phenotyping',
 'chondroid tissue',
 'pasteurella infections',
 'and encephalitis',
 'maxillary sinus cancer',
 'oral iron supplementation',
 'salmonella enterica',
 'topographical orientation',
 'giant cystic pheochromocytoma',
 'beta lactam antibiotics',
 'expert presentations',
 'paraneoplastic choreoathetosis',
 'liquid nitrogen autologous graft',
 'cerebrospinal fluid diversion',
 'petrous aneurysm',
 'liver torsion',
 'total aortic arch replacement',
 'ganglionic nicotinic acetylcholine receptor',
 'childhood ataxia with central nervous system hypomyelination/vanishing white matter',
 'avermectin and pyridine',
 'pulmonary hepatoid adenocarcinoma',
 'progressive myelopathy',
 'glucocorticoids/therapeutic use',
 'aspergillus infection',
 'pharmacy recruit

In [2]:
# Check relevant MeSH terms for tropical/parasitic/infectious diseases
tropical_keywords_to_check = [
    "malaria", "dengue", "tuberculosis", "melioidosis", 
    "leishmania", "schistosom", "trypanosom", "filariasis", 
    "typhus", "leptospirosis", "chikungunya", "zika"
]

matching_mesh = [
    m for m in mdc.mesh_term_list 
    if any(k in m.lower() for k in tropical_keywords_to_check)
]

print(f"Found {len(matching_mesh)} matching MeSH terms:")
print(matching_mesh[:15])

Found 332 matching MeSH terms:
['Leishmaniasis, Cutaneous / immunology', 'Zika Virus Infection / blood', 'Tuberculosis, Female Genital / diagnostic imaging', 'Tuberculosis, Lymph Node / complications', 'Leptospirosis / pathology', 'Tuberculosis, Osteoarticular / diagnosis', 'Tuberculosis, Osteoarticular / drug therapy', 'Dengue Virus', 'Leptospirosis / epidemiology', 'Tuberculosis / epidemiology', 'Mycobacterium tuberculosis / genetics', 'Tuberculosis, Pulmonary / metabolism', 'Leishmania / immunology', 'Leishmaniasis, Visceral / genetics', 'Leishmania tropica / genetics']


In [ ]:
# tropical_disease_filter = [
#     # --------------------------------------------------------------------------
#     # 1. MeSH TERMS: Captures indexed descriptors and all /qualifier subheadings
#     # --------------------------------------------------------------------------
#     {
#         'field': 'mesh_terms',
#         'string_list': [
#             # Mycobacterial / Bacterial
#             'Tuberculosis', 'Mycobacterium tuberculosis',
#             'Melioidosis', 'Burkholderia pseudomallei',
#             'Leprosy', 'Mycobacterium leprae',
#             'Leptospirosis', 'Leptospira',
#             'Typhus', 'Orientia tsutsugamushi', 'Rickettsia',
#             'Buruli Ulcer',

#             # Viral Tropical Infections
#             'Dengue', 'Severe Dengue', 'Dengue Virus',
#             'Zika Virus', 'Zika Virus Infection',
#             'Chikungunya', 'Chikungunya Fever',
#             'Yellow Fever', 'Rabies',

#             # Protozoan / Parasitic
#             'Malaria', 'Antimalarials', 'Plasmodium',
#             'Leishmania', 'Leishmaniasis',
#             'Amebiasis', 'Entamoeba histolytica',
#             'Trypanosomiasis', 'Chagas Disease',

#             # Helminthic / Flukes
#             'Schistosoma', 'Schistosomiasis',
#             'Filariasis', 'Elephantiasis',
#             'Echinococcosis', 'Cysticercosis',
#             'Strongyloidiasis', 'Ascariasis'
#         ],
#         'operator': 'any',        # Matches if at least one term is present[cite: 2]
#         'match_type': 'partial_match'  # Substring match captures '/ diagnosis', etc.[cite: 2]
#     },

#     # --------------------------------------------------------------------------
#     # 2. KEYWORDS: Catches author-specific jargon, acronyms, and pathogen names
#     # --------------------------------------------------------------------------
#     {
#         'field': 'keywords',
#         'string_list': [
#             'tropical disease', 'neglected tropical',
#             'tuberculosis', 'tb', 'pulmonary tb', 'extrapulmonary tb',
#             'melioidosis', 'whitmore',
#             'dengue', 'dengue hemorrhagic fever', 'dhf',
#             'zika', 'chikungunya',
#             'malaria', 'falciparum', 'vivax',
#             'leishmania', 'kala-azar', 'black fever',
#             'leptospira', 'leptospirosis',
#             'scrub typhus', 'schistosomiasis'
#         ],
#         'operator': 'any',
#         'match_type': 'partial_match'
#     },

#     # --------------------------------------------------------------------------
#     # 3. CLINICAL CASE NARRATIVE: Catches in-text diagnoses and microbiological findings
#     # --------------------------------------------------------------------------
#     {
#         'field': 'case_strings',
#         'string_list': [
#             'tuberculosis', 'mycobacterium',
#             'melioidosis', 'burkholderia pseudomallei',
#             'dengue', 'dengue shock syndrome',
#             'plasmodium falciparum', 'plasmodium vivax',
#             'leishmania', 'donovani',
#             'leptospira', 'leptospirosis',
#             'scrub typhus', 'orientia tsutsugamushi',
#             'schistosoma'
#         ],
#         'operator': 'any'
#     }
# ]

In [ ]:
# mdc.create_dataset(
#     dataset_name='tropical_infectious_diseases_cohort',
#     filter_list=tropical_disease_filter,
#     dataset_type='multimodal'
# )

# print(f"Total Patients Extracted: {len(mdc.filtered_cases):,}")
# print(f"Total Images Extracted:   {len(mdc.filtered_image_metadata_df):,}")

The tropical_infectious_diseases_cohort was successfully created!
Total Patients Extracted: 2,893
Total Images Extracted:   6,082


In [ ]:
# comprehensive_tropical_case_filter = [
#     {
#         'field': 'case_strings',
#         'operator': 'any',
#         'string_list': [
#             # ------------------------------------------------------------------
#             # 1. MYCOBACTERIAL & TROPICAL BACTERIAL ZOONOSES
#             # ------------------------------------------------------------------
#             'tuberculosis', 'mycobacterium', 'tuberculoma', 'acid-fast', 'ziehl-neelsen',
#             'pott disease', 'scrofula', 'ghon focus',
#             'melioidosis', 'burkholderia', 'whitmore',
#             'leptospir', 'weil disease',
#             'scrub typhus', 'tsutsugamushi', 'orientia', 'eschar',
#             'leprosy', 'mycobacterium leprae', 'hansen disease', 'lepromatous',
#             'buruli ulcer', 'mycobacterium ulcerans', 'bairnsdale ulcer',
#             'rickettsia', 'murine typhus', 'spotted fever',
#             'borrelia', 'relapsing fever', 'treponema pertenue', 'yaws',
#             'yersinia pestis', 'bubonic plague',

#             # ------------------------------------------------------------------
#             # 2. ARBOVIRUSES & TROPICAL VIRAL FEVERS
#             # ------------------------------------------------------------------
#             'dengue', 'dhf', 'dss', 'severe dengue', 'breakbone',
#             'zika', 'microcephaly', 'congenital zika',
#             'chikungunya',
#             'yellow fever',
#             'japanese encephalitis',
#             'crimean-congo', 'lassa fever', 'ebola', 'marburg',
#             'oropouche', 'kyasanur', 'alkhurma',

#             # ------------------------------------------------------------------
#             # 3. PROTOZOAL INFECTIONS
#             # ------------------------------------------------------------------
#             'malaria', 'plasmodium', 'falciparum', 'vivax', 'malariae', 'ovale', 'knowlesi',
#             'schizont', 'gametocyte', 'ring-form trophozoite', 'blackwater fever',
#             'leishmania', 'leishmaniasis', 'kala-azar', 'kala azar', 'donovani',
#             'amastigote', 'ld bodies', 'leishman-donovan', 'cutaneous leishmaniasis',
#             'entamoeba histolytica', 'amoebiasis', 'amebiasis', 'amoebic liver abscess', 'amebic abscess',
#             'trypanosoma', 'trypanosomiasis', 'chagas', 'chagasic', 'cruzi', 'romana sign',
#             'sleeping sickness', 'brucei',

#             # ------------------------------------------------------------------
#             # 4. TISSUE PARASITES, FLUKES & HELMINTHS (High Imaging Yield)
#             # ------------------------------------------------------------------
#             'cysticercosis', 'neurocysticercosis', 'taenia solium', 'cysticerci', 'scolex',
#             'echinococcus', 'echinococcosis', 'hydatid', 'hydatid cyst', 'water lily sign',
#             'schistosoma', 'schistosomiasis', 'bilharzia', 'symmers fibrosis',
#             'paragonimiasis', 'paragonimus', 'pulmonary distomiasis',
#             'fasciola', 'fascioliasis', 'clonorchis', 'clonorchiasis', 'opisthorchis',
#             'filariasis', 'filarial', 'wuchereria bancrofti', 'brugia', 'elephantiasis',
#             'filarial dance', 'loiasis', 'loa loa', 'onchocerca', 'onchocerciasis', 'river blindness',
#             'strongyloides', 'strongyloidiasis', 'hyperinfection syndrome',
#             'dracunculus', 'dracunculiasis', 'guinea worm',
#             'gnathostoma', 'gnathostomiasis', 'angiostrongylus',

#             # ------------------------------------------------------------------
#             # 5. ENDEMIC DEEP & SUBCUTANEOUS TROPICAL MYCOSES
#             # ------------------------------------------------------------------
#             'mycetoma', 'madura foot', 'madurella', 'actinomycetoma', 'eumycetoma', 'grains',
#             'chromoblastomycosis', 'fonsecaea', 'medlar bodies', 'muriform cells',
#             'talaromyces', 'talaromycosis', 'penicillium marneffei',
#             'paracoccidioidomycosis', 'blastomyces', 'lobomycosis', 'lacazia loboi',
#             'rhinosporidiosis', 'rhinosporidium',
#             'histoplasmosis', 'histoplasma capsulatum', 'sporotrichosis', 'sporothrix'
#         ]
#     }
# ]

In [ ]:
# mdc.create_dataset(
#     dataset_name='tropical_infectious_diseases_cohort_case_text_only',
#     filter_list=comprehensive_tropical_case_filter,
#     dataset_type='multimodal'
# )

# print(f"Total Patients Extracted: {len(mdc.filtered_cases):,}")
# print(f"Total Images Extracted:   {len(mdc.filtered_image_metadata_df):,}")

The tropical_infectious_diseases_cohort_case_text_only was successfully created!
Total Patients Extracted: 7,874
Total Images Extracted:   12,305


In [4]:
cleaned_tropical_case_filter = [
    {
        'field': 'case_strings',
        'operator': 'any',
        'string_list': [
            # ------------------------------------------------------------------
            # 1. MYCOBACTERIAL & BACTERIAL ZOONOSES
            # ------------------------------------------------------------------
            'tuberculosis', 'mycobacterium tuberculosis', 'tuberculoma', 'm. tuberculosis',
            'pott disease', 'scrofula', 'ghon focus',
            'melioidosis', 'burkholderia pseudomallei', 'whitmore disease',
            'leptospir', 'weil disease',
            'scrub typhus', 'tsutsugamushi', 'orientia', 'eschar',
            'leprosy', 'mycobacterium leprae', 'hansen disease', 'lepromatous leprosy',
            'buruli ulcer', 'mycobacterium ulcerans', 'bairnsdale ulcer',
            'rickettsia', 'murine typhus', 'spotted fever',
            'borrelia', 'relapsing fever', 'treponema pertenue', 'yaws',
            'yersinia pestis', 'bubonic plague',

            # ------------------------------------------------------------------
            # 2. ARBOVIRUSES & HEMORRHAGIC FEVERS
            # ------------------------------------------------------------------
            'dengue', 'dengue hemorrhagic fever', 'dengue shock syndrome', 'dhf', 'dss',
            'zika', 'congenital zika',
            'chikungunya',
            'yellow fever',
            'japanese encephalitis',
            'crimean-congo hemorrhagic', 'lassa fever', 'ebola virus', 'marburg virus',
            'oropouche', 'kyasanur forest',

            # ------------------------------------------------------------------
            # 3. PROTOZOAL INFECTIONS (Safely Disambiguated)
            # ------------------------------------------------------------------
            'malaria', 'plasmodium', 'falciparum', 'vivax', 'plasmodium ovale', 'p. ovale', 'knowlesi',
            'malarial parasite', 'blackwater fever', 'ring-form trophozoite',
            'leishmania', 'leishmaniasis', 'kala-azar', 'kala azar', 'donovani',
            'amastigote', 'ld bodies', 'leishman-donovan', 'cutaneous leishmaniasis',
            'entamoeba histolytica', 'amoebiasis', 'amebiasis', 'amoebic liver abscess', 'amebic abscess',
            'trypanosoma', 'trypanosomiasis', 'chagas disease', 'trypanosoma cruzi', 'romana sign',
            'sleeping sickness', 'trypanosoma brucei',

            # ------------------------------------------------------------------
            # 4. TISSUE PARASITES, FLUKES & HELMINTHS (High Imaging Yield)
            # ------------------------------------------------------------------
            'cysticercosis', 'neurocysticercosis', 'taenia solium', 'cysticerci', 'cysticercus',
            'echinococcus', 'echinococcosis', 'hydatid cyst', 'water lily sign',
            'schistosoma', 'schistosomiasis', 'bilharzia', 'symmers fibrosis',
            'paragonimiasis', 'paragonimus',
            'fasciola', 'fascioliasis', 'clonorchis', 'clonorchiasis', 'opisthorchis',
            'filariasis', 'filarial dance', 'wuchereria bancrofti', 'brugia malayi', 'elephantiasis',
            'loiasis', 'loa loa', 'onchocerca', 'onchocerciasis', 'river blindness',
            'strongyloides', 'strongyloidiasis', 'hyperinfection syndrome',
            'dracunculus', 'dracunculiasis', 'guinea worm',
            'gnathostoma', 'gnathostomiasis', 'angiostrongylus',

            # ------------------------------------------------------------------
            # 5. ENDEMIC DEEP & SUBCUTANEOUS TROPICAL MYCOSES
            # ------------------------------------------------------------------
            'mycetoma', 'madura foot', 'madurella', 'actinomycetoma', 'eumycetoma',
            'chromoblastomycosis', 'fonsecaea', 'medlar bodies', 'muriform cells',
            'talaromyces', 'talaromycosis', 'penicillium marneffei',
            'paracoccidioidomycosis', 'blastomyces', 'blastomycosis', 'lobomycosis', 'lacazia loboi',
            'rhinosporidiosis', 'rhinosporidium',
            'histoplasmosis', 'histoplasma capsulatum', 'sporotrichosis', 'sporothrix'
        ]
    }
]

In [5]:
mdc.create_dataset(
    dataset_name="clean_tropical_cohort",
    filter_list=cleaned_tropical_case_filter,
    dataset_type="multimodal",
)

The clean_tropical_cohort was successfully created!


In [6]:
print(f"Total Patients Extracted: {len(mdc.filtered_cases):,}")
print(f"Total Images Extracted:   {len(mdc.filtered_image_metadata_df):,}")

Total Patients Extracted: 5,990
Total Images Extracted:   9,403
